In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

if os.environ['AICREDITS_API_KEY']:
    
    print("AICredits API Key is set.")
else:
    raise ValueError("AICredits API Key is not set.")

AICredits API Key is set.


In [18]:
llm = ChatOpenAI(
    model="openai/gpt-5-nano",
    base_url="https://api.aicredits.in/v1",
    api_key=os.environ["AICREDITS_API_KEY"],
)

## **TOOLS**
### **DuckDuckGo Search Tool**

In [19]:
from langchain_core.tools import tool
from duckduckgo_search import DDGS

@tool
def search_duckduckgo(query: str) -> str:
    """This tool searches the latest news on DuckDuckGo for the given query and returns the results."""
    with DDGS() as ddgs:
        duck_search = ddgs()
        return duck_search.invoke(query)

### **Arxiv Query Tool**

In [20]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

@tool
def arxiv_tool(query: str) -> str:
    """This tool allows you to query the arXiv database for research papers"""
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(query)

### **Wikipedia Search Tool**

In [21]:

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(query: str):

    """This tool allows you to search Wikipedia for information on a given topic."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

### **Custom Tool**

In [22]:
@tool
def personal_info(name: str):
    """Use this tool to get personal information about Alice, Bob or Charlie."""
    info = {
        "Alice": "Alice is a software engineer with 5 years of experience in AI.",
        "Bob": "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups." 
    }
    return info.get(name, "No information available for this person.")

### **TOOL BINDING**

In [23]:
tools = [search_duckduckgo, arxiv_tool, wiki_tool, personal_info]
llm_with_tools = llm.bind_tools(tools)

In [28]:
response = llm_with_tools.invoke("What is the news and research about Solar Cells ")
response.tool_calls

[{'name': 'search_duckduckgo',
  'args': {'query': 'solar cells latest news'},
  'id': 'call_JTXpcVhdr6o998RFgppEetG8',
  'type': 'tool_call'},
 {'name': 'arxiv_tool',
  'args': {'query': 'solar cells'},
  'id': 'call_HQPj4sXKsvWyceBsSfqWveOu',
  'type': 'tool_call'},
 {'name': 'wiki_tool',
  'args': {'query': 'Solar cell'},
  'id': 'call_LMoQytw363tLwemMvNDgdWR3',
  'type': 'tool_call'}]